In [1]:
import pandas as pd

In [32]:
import ast
import pandas as pd

# File path
file_path = 'C:\\Users\\Dell\\OneDrive\\Desktop\\Final-player.txt'

# Read the content of the file
with open(file_path, 'r', encoding='utf-8') as file:
    content = file.read()

# Evaluate the content to convert it to a Python object
data = ast.literal_eval(content)

# Extract the data into a format suitable for DataFrame
parsed_data = []
for row in data:
    player_id, name, club, club_value, birth, weight, height, country, role, foot, transfers, injuries = row
    transfers_str = ', '.join([f'{x[0]} -> {x[1]}, {x[2]}, {x[3]}' for x in transfers])
    injuries_str = ', '.join([f'{x[0]}, {x[1]}, {x[2]}' for x in injuries])
    parsed_data.append([player_id, name, club, club_value, birth, weight, height, country, role, foot, transfers_str, injuries_str])

# Create a DataFrame
columns = ['PlayerId', 'name', 'club', 'club_value', 'birth', 'weight', 'height', 'country', 'role', 'foot', 'transfers', 'injuries']
df_soccer = pd.DataFrame(parsed_data, columns=columns)

print(df_soccer.head())


  PlayerId             name             club    club_value       birth weight  \
0   238223          Ederson  Manchester City  1,14 Bill. €  1993-08-17     89   
1    40423    Claudio Bravo  Manchester City  1,14 Bill. €  1983-04-13     84   
2   371021   Arijanet Murić  Manchester City  1,14 Bill. €  1998-11-07      -   
3   186590      John Stones  Manchester City  1,14 Bill. €  1994-05-28     69   
4   176553  Aymeric Laporte  Manchester City  1,14 Bill. €  1994-05-27     85   

  height  country                    role   foot  \
0  1,88    Brazil              Goalkeeper   left   
1  1,84     Chile              Goalkeeper  right   
2  1,98    Kosovo              Goalkeeper  right   
3  1,88   England  Defender - Centre-Back  right   
4  1,91    France  Defender - Centre-Back   left   

                                           transfers  \
0  \nBenfica  -> \nMan City , 17/18, Jul 1, 2017,...   
1  \nFC Barcelona  -> \nMan City , 16/17, Aug 25,...   
2  \nNAC Breda  -> \nMan City , 

In [33]:
# Exporting the DataFrame to a CSV file
df_soccer.to_csv('df_soccer.csv', index=False)


In [34]:
df_soccer.head()

,PlayerId,name,club,club_value,birth,weight,height,country,role,foot,transfers,injuries
0,238223,Ederson,Manchester City,"1,14 Bill. €",1993-08-17,89,"1,88",Brazil,Goalkeeper,left,"\nBenfica -> \nMan City , 17/18, Jul 1, 2017,...","16/17, Meniscal Injury, 29"
1,40423,Claudio Bravo,Manchester City,"1,14 Bill. €",1983-04-13,84,"1,84",Chile,Goalkeeper,right,"\nFC Barcelona -> \nMan City , 16/17, Aug 25,...","18/19, Achilles tendon rupture, 314, 16/17, Di..."
2,371021,Arijanet Murić,Manchester City,"1,14 Bill. €",1998-11-07,-,"1,98",Kosovo,Goalkeeper,right,"\nNAC Breda -> \nMan City , 18/19, Aug 22, 20...",
3,186590,John Stones,Manchester City,"1,14 Bill. €",1994-05-28,69,"1,88",England,Defender - Centre-Back,right,"\nEverton -> \nMan City , 16/17, Aug 9, 2016,...","18/19, Muscle Injury, 6, 17/18, Minor Knock, 4..."
4,176553,Aymeric Laporte,Manchester City,"1,14 Bill. €",1994-05-27,85,"1,91",France,Defender - Centre-Back,left,"\nAthletic -> \nMan City , 17/18, Jan 30, 201...","18/19, Muscle Injury, 12, 16/17, Groin Injury,..."


# NBA data--webscrape


In [4]:
!Pip install html5lib

##importing packages needed for scrape
from urllib.request import urlopen
from bs4 import BeautifulSoup
import pandas as pd
import time
import requests
import warnings

##function designed to scrape all of the injuries reported in NBA history 
def single(tot):
    ##number that determines where the start is, or in other words what page of the table the url is 
    ##as it increments by 25, page 1 => start=0 page 2 => start=25 page 3 => start=50 etc
    num = 0
    ##supress warnining for using df.append rather than pd.concat
    warnings.filterwarnings('ignore', message='The frame.append method is deprecated')
    while tot != 0:
        if num == 0:
            url = f'https://www.prosportstransactions.com/basketball/Search/SearchResults.php?Player=&Team=&BeginDate=&EndDate=&ILChkBx=yes&Submit=Search&start={num}'
            response = requests.get(url)
            html = response.content
            soup = BeautifulSoup(html, 'html5lib')
            table = soup.find('table', attrs={'class': 'datatable center'})
            ##here I create the df that will be populated throughout the scrape
            df = pd.read_html(str(table))[0]
            num = num + 25           ##increment to 25 to move onto page 2
            tot = tot - 1            ##reduce tot to see how many pages are left to visit
            time.sleep(2.38)         ##rate limit to get the total corpus(n=1507) in 1 hour (≈ 25.21 requests/minute)
        else:
            url = f'https://www.prosportstransactions.com/basketball/Search/SearchResults.php?Player=&Team=&BeginDate=&EndDate=&ILChkBx=yes&Submit=Search&start={num}'
            response = requests.get(url)
            html = response.content
            soup = BeautifulSoup(html, 'html5lib')
            table = soup.find('table', attrs={'class': 'datatable center'})
            ##here I append to the df that was created 
            df = df.append(pd.read_html(str(table))[0])
            num = num + 25           ##increment the start to go to the next page
            tot = tot - 1            ##reduce the total to determine number of remaining pages
            time.sleep(2.38)         ##rate limit to get the total corpus(n=1507) in 1 hour (≈ 25.21 requests/minute)

    df.columns = ['Date','Team','Acquired','Relinquished','Notes']
    df = df[df['Date']!='Date']
    df.reset_index(drop=True, inplace=True)
    df.Acquired = df.Acquired.str.replace('•','') # remove • in front of player's name
    df.Relinquished = df.Relinquished.str.replace('•','') # remove • in front of player's name
    df['Date'] = pd.to_datetime(df['Date'].str.strip(), format='%Y-%m-%d')
    return df

##calling for all NBA seasons data except current(1949-50 until 2021-22) as current season is not yet finished
df = multiple(2021,2022)

df = single(1507)

In [12]:
!pip install requests
!pip install beautifulsoup4
!pip install pandas


In [13]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

def single(tot):
    frames = []  # List to store DataFrames
    for num in range(0, tot * 25, 25):
        url = f'https://www.prosportstransactions.com/basketball/Search/SearchResults.php?Player=&Team=&BeginDate=&EndDate=&ILChkBx=yes&Submit=Search&start={num}'
        response = requests.get(url)
        html = response.content
        soup = BeautifulSoup(html, 'html.parser')
        table = soup.find('table', attrs={'class': 'datatable center'})

        if table is not None:
            # Convert table to DataFrame
            df = pd.read_html(str(table))[0]
            # Process the DataFrame (e.g., renaming columns, removing unwanted rows) here
            frames.append(df)

        # Sleep to avoid overloading the server
        time.sleep(2.38)

    # Concatenate all the DataFrames
    result_df = pd.concat(frames, ignore_index=True)
    
    # Clean and process the DataFrame
    result_df.columns = ['Date', 'Team', 'Acquired', 'Relinquished', 'Notes']
    result_df = result_df[result_df['Date'] != 'Date']
    result_df.reset_index(drop=True, inplace=True)
    result_df['Acquired'] = result_df['Acquired'].str.replace('•', '') # remove • in front of player's name
    result_df['Relinquished'] = result_df['Relinquished'].str.replace('•', '') # remove • in front of player's name
    result_df['Date'] = pd.to_datetime(result_df['Date'].str.strip(), format='%Y-%m-%d')

    return result_df

# Change 1511 to the number of pages you want to scrape
df = single(1511)


##exporting df as csv
#df.to_csv('NBA Player Stats(1950 - 2022).csv', index = True)

In [14]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

def single(tot):
    frames = []  # List to store DataFrames
    for num in range(0, tot * 25, 25):
        url = f'https://www.prosportstransactions.com/basketball/Search/SearchResults.php?Player=&Team=&BeginDate=&EndDate=&ILChkBx=yes&Submit=Search&start={num}'
        response = requests.get(url)
        html = response.content
        soup = BeautifulSoup(html, 'html.parser')
        table = soup.find('table', attrs={'class': 'datatable center'})

        if table is not None:
            # Convert table to DataFrame
            df = pd.read_html(str(table))[0]
            # Process the DataFrame (e.g., renaming columns, removing unwanted rows) here
            frames.append(df)

        # Sleep to avoid overloading the server
        time.sleep(2.38)

    # Concatenate all the DataFrames
    result_df = pd.concat(frames, ignore_index=True)
    
    # Clean and process the DataFrame
    result_df.columns = ['Date', 'Team', 'Acquired', 'Relinquished', 'Notes']
    result_df = result_df[result_df['Date'] != 'Date']
    result_df.reset_index(drop=True, inplace=True)
    result_df['Acquired'] = result_df['Acquired'].str.replace('•', '') # remove • in front of player's name
    result_df['Relinquished'] = result_df['Relinquished'].str.replace('•', '') # remove • in front of player's name
    result_df['Date'] = pd.to_datetime(result_df['Date'].str.strip(), format='%Y-%m-%d')

    return result_df

# Change 1511 to the number of pages you want to scrape
df = single(1511)


df.head(1)

# df soccer

In [25]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

def single(tot):
    frames = []  # List to store DataFrames
    for num in range(0, tot * 25, 25):
        url = f'https://www.prosportstransactions.com/soccer/Search/SearchResults.php?Player=&Team=&BeginDate=1990-06-01&EndDate=2023-06-01&ILChkBx=yes&InjuriesChkBx=yes&submit=Search&start=0={num}'
        response = requests.get(url)
        html = response.content
        soup = BeautifulSoup(html, 'html.parser')
        table = soup.find('table', attrs={'class': 'datatable center'})

        if table is not None:
            # Convert table to DataFrame
            df = pd.read_html(str(table))[0]
            # Process the DataFrame (e.g., renaming columns, removing unwanted rows) here
            frames.append(df)

        # Sleep to avoid overloading the server
        time.sleep(2.38)

    # Concatenate all the DataFrames
    result_df = pd.concat(frames, ignore_index=True)
    
    # Clean and process the DataFrame
    result_df.columns = ['Date', 'Team', 'Acquired', 'Relinquished', 'Notes']
    result_df = result_df[result_df['Date'] != 'Date']
    result_df.reset_index(drop=True, inplace=True)
    result_df['Acquired'] = result_df['Acquired'].str.replace('•', '') # remove • in front of player's name
    result_df['Relinquished'] = result_df['Relinquished'].str.replace('•', '') # remove • in front of player's name
    result_df['Date'] = pd.to_datetime(result_df['Date'].str.strip(), format='%Y-%m-%d')

    return result_df

# Change 1511 to the number of pages you want to scrape
df = single(125)


In [26]:
df.head(100)

,Date,Team,Acquired,Relinquished,Notes
0,1996-08-22,Revolution,NaN,Patrick Tardieu,placed on IR
1,1997-03-19,Tornado (NASL),NaN,Brian Johnson,placed on IR
2,1997-03-19,Wizards,NaN,Scott Uderitz,placed on IR
3,1997-03-29,D.C. United,NaN,Brian Kamler,placed on IR
4,1997-03-29,D.C. United,NaN,Mario Gori,placed on IR
...,...,...,...,...,...
95,2006-09-18,Fire,NaN,Jeff Curtin,placed on IL
96,2008-07-01,Chivas USA,NaN,Raphael Wicky,surgery on ankle (out indefinitely) (date appr...
97,2008-09-10,Galaxy,NaN,Charles Alamo,placed on IR
98,2008-09-10,Galaxy,NaN,Michael Gavin,placed on disabled list


In [27]:
df.shape

(3125, 5)

In [28]:
# Drop the "Acquired" column
df = df.drop(columns=['Acquired'])

# Rename the "Relinquished" column to "Name"
df = df.rename(columns={'Relinquished': 'Name'})

# Print the first few rows to verify
print(df.head())


        Date            Team              Name         Notes
0 1996-08-22      Revolution   Patrick Tardieu  placed on IR
1 1997-03-19  Tornado (NASL)     Brian Johnson  placed on IR
2 1997-03-19         Wizards     Scott Uderitz  placed on IR
3 1997-03-29     D.C. United      Brian Kamler  placed on IR
4 1997-03-29     D.C. United        Mario Gori  placed on IR
